# Figure 1 — Resource cost: power-law QPE vs logarithmic Rodeo

**Paper location:** Section 6 (Benchmark simulation), Fig. 1.

## What this figure shows

The central complexity claim of the paper, made concrete on the single-spin
driven–dissipative model of Ramusat & Savona. We plot the **controlled-evolution
depth** required by each filter to reach a target filtering error `ε`, against the
number of target digits `log₁₀(1/ε)`.

- **Phase estimation (QPE)** isolates the zero sector by resolving eigenvalues with
  a finite phase register. Its leakage scales as `1/(g²T²)`, so the depth grows as a
  **power law** `T ~ ε^{-1/2}` — a *straight line* on the semi-log panel (a) and an
  *explosion* on the linear panel (b).
- **Rodeo (deterministic)** fixes the resolution scale at `~1/g` and suppresses the
  residual weight by repeated measurement-conditioned cycles. Its depth grows only
  **logarithmically**, `T ~ log(1/ε)` — a *concave* curve in (a), *linear* in (b).
- **Rodeo (Gaussian)** is the random-time baseline; we plot its analytic expected
  residual weight. It shares the logarithmic scaling but with worse constants.

## Model & comparison

| | |
|---|---|
| **Model** | single spin, `H = h σ_x`, jump `A = σ⁻`; Liouvillian gap `g = 1/2` for all `h` |
| **Compared** | QPE register (best admissible at each depth) vs Rodeo van der Corput schedule (optimized over cycle count) |
| **Cost metric** | worst-case residual weight `max_{j≠0,1} |·|²` of a nonzero embedding mode |
| **Key number** | QPE exponent fit ≈ 0.50 (matches `ε^{-1/2}`); Rodeo `R² ≈ 1` linear in `log(1/ε)` |

The crossover location depends on the constants (`t₀`, schedule scale `c`); the
*scaling exponents* do not. Because `g = 1/2` for every `h`, the fitted exponents are
identical for `h = 0.5, 1.0, 1.5` — confirming the scaling is set by the spectral
separation, not the field.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../reproduce/core"))

import numpy as np
import matplotlib.pyplot as plt

from cost_compare import (
    G, T0,
    qpe_epsilon, qpe_depth,
    rodeo_epsilon_gaussian, rodeo_depth_gaussian,
    rodeo_epsilon_deterministic, rodeo_depth_deterministic,
)

H = 0.5          # field value for the single-spin model
C_GAUSS = 2.0    # Gaussian t_rms = C/g (near depth-optimal)
T_RMS = C_GAUSS / G
A_DET = 3.0      # deterministic schedule scale
print(f"single-spin gap g = {G}, QPE time scale t0 = {T0}")


single-spin gap g = 0.5, QPE time scale t0 = 0.2


### Compute depth vs target precision
Each filter's error is computed exactly from the embedding spectrum; we then read off the controlled-evolution depth at each achievable precision.

In [2]:
# QPE: sweep phase-register size
qe = np.array([qpe_epsilon(H, t) for t in range(2, 16)])
qT = np.array([qpe_depth(t)       for t in range(2, 16)])

# Rodeo: sweep number of cycles
ns = np.arange(1, 31)
dT = np.array([rodeo_depth_deterministic(n, A_DET)   for n in ns])
de = np.array([rodeo_epsilon_deterministic(H, n, A_DET) for n in ns])
gT = np.array([rodeo_depth_gaussian(n, T_RMS)        for n in ns])
ge = np.array([rodeo_epsilon_gaussian(H, n, T_RMS)   for n in ns])

prec = lambda e: np.log10(1/np.clip(e, 1e-300, None))
qp, dp, gp = prec(qe), prec(de), prec(ge)

# fitted scalings
cq = np.polyfit(qp, np.log10(qT), 1)   # QPE: log T vs precision -> exponent
cr = np.polyfit(dp, dT, 1)             # Rodeo: T vs precision -> linear
print(f"QPE exponent  : T ~ eps^(-{cq[0]:.3f})   (expected 0.5)")
print(f"Rodeo linear  : T = {cr[0]:.1f} * log10(1/eps) + {cr[1]:.1f}")


QPE exponent  : T ~ eps^(-0.501)   (expected 0.5)
Rodeo linear  : T = 6.7 * log10(1/eps) + 2.9


### Figure
Improved styling: colour-blind-friendly palette, shared legend, fit lines, and clear power-law-vs-log contrast across the two panels.

In [ ]:
C_QPE, C_RG, C_RD = "#0072B2", "#D55E00", "#009E73"

fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 4.4))

# (a) semi-log: power law = straight, log = concave
axa.scatter(qp, qT, s=26, color=C_QPE, zorder=3, label="QPE")
axa.scatter(gp, gT, s=22, marker="s", color=C_RG, zorder=3, label="Rodeo (Gauss.)")
axa.scatter(dp, dT, s=26, marker="^", color=C_RD, zorder=3, label="Rodeo (det.)")
xq = np.linspace(qp.min(), qp.max(), 50)
axa.plot(xq, 10**(cq[0]*xq + cq[1]), "-", color=C_QPE, lw=1.0, alpha=0.7)
xd = np.linspace(dp.min(), dp.max(), 50)
axa.plot(xd, cr[0]*xd + cr[1], "-", color=C_RD, lw=1.0, alpha=0.7)
axa.set_yscale("log"); axa.set_ylim(0.3, 3e6)
axa.set_xlabel(r"target precision $\log_{10}(1/\varepsilon)$")
axa.set_ylabel(r"controlled-evolution depth $T$")
axa.set_title("(a) semi-log: QPE power law vs Rodeo log")
axa.legend(loc="upper left"); axa.grid(alpha=0.25, which="both")

# (b) linear: Rodeo straight, QPE explodes
axb.plot(qp, qT, "o-", color=C_QPE, label="QPE")
axb.plot(gp, gT, "s-", color=C_RG, label="Rodeo (Gauss.)")
axb.plot(dp, dT, "^-", color=C_RD, label="Rodeo (det.)")
axb.set_xlim(0, 9); axb.set_ylim(0, 650)
axb.set_xlabel(r"target precision $\log_{10}(1/\varepsilon)$")
axb.set_ylabel(r"controlled-evolution depth $T$")
axb.set_title("(b) linear: QPE diverges, Rodeo linear")
axb.legend(loc="upper left"); axb.grid(alpha=0.25)

fig.tight_layout()
fig.savefig("fig_cost.pdf", bbox_inches="tight")
plt.show()